<a href="https://colab.research.google.com/github/HIMESH-RINCHHODIYA/google_ai/blob/main/ATTENDENCE_MANAGEMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras-facenet
!pip install mtcnn
!pip install opencv-python
!pip install pandas
!pip install scikit-learn
!pip install retina-face

In [ ]:
import os
import cv2
import pickle
import numpy as np
import pandas as pd

from retinaface import RetinaFace
from mtcnn import MTCNN
from keras_facenet import FaceNet
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!pip install -U lz4 joblib mtcnn -q
detector = MTCNN()

In [ ]:
#   Block 3: Initialize FaceNet

embedder = FaceNet()

print("FaceNet Loaded Successfully")

FaceNet Loaded Successfully


In [ ]:
#Block 4: Mount Google Drive


from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATASET_PATH = "/content/drive/MyDrive/collab/UPLOAD PHOTO (File responses)-20260617T044741Z-3-001"

print("Exists:", os.path.exists(DATASET_PATH))

print("\nFiles/Folders inside:\n")

for item in os.listdir(DATASET_PATH)[:20]:
    print(item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exists: True

Files/Folders inside:

UPLOAD PHOTO (File responses)


In [ ]:
###Block 5: Dataset Path

###Change according to your folder.

DATASET_PATH = "/content/drive/MyDrive/collab/UPLOAD PHOTO (File responses)-20260617T044741Z-3-001/UPLOAD PHOTO (File responses)"

In [ ]:
# ==================================
# Face Detection Function (FIXED)
# ==================================

def detect_face(img_path):

    img = cv2.imread(img_path)

    if img is None:
        raise ValueError(
            f"Cannot read image: {img_path}"
        )

    rgb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    faces = detector.detect_faces(rgb)

    # No face found
    if len(faces) == 0:
        raise ValueError(
            "No face detected"
        )

    # Highest confidence face
    best_face = max(
        faces,
        key=lambda x: x['confidence']
    )

    x, y, w, h = best_face['box']

    # Fix negative coordinates
    x = max(0, x)
    y = max(0, y)

    # Ensure valid box
    if w <= 0 or h <= 0:
        raise ValueError(
            "Invalid face dimensions"
        )

    # Crop face
    face = rgb[
        y:y+h,
        x:x+w
    ]

    # Empty crop protection
    if face.size == 0:
        raise ValueError(
            "Empty face crop"
        )

    # Face too small
    if face.shape[0] < 30 or face.shape[1] < 30:
        raise ValueError(
            "Face too small"
        )

    face = cv2.resize(
        face,
        (160,160)
    )

    return face

In [ ]:
###Block 6: Generate Embedding Function

def get_embedding(img_path):

    face = detect_face(img_path)

    face = np.expand_dims(
        face,
        axis=0
    )

    embedding = embedder.embeddings(face)

    return embedding[0]

In [ ]:
# Extract student name from filename

import os

def extract_name(filename):

    name = os.path.splitext(filename)[0]

    # Handle filenames like:
    # 12345 - RAHUL GANDHI.jpg
    if " - " in name:
        name = name.split(" - ")[-1]

    # Remove _1, _2, _3, etc.
    if "_" in name:

        parts = name.split("_")

        if parts[-1].isdigit():
            name = "_".join(parts[:-1])

    return name.strip().upper()

In [ ]:
# Block 7: Create Student Embedding Database

database = {}
count = 0

for file in os.listdir(DATASET_PATH):

    if file.lower().endswith(
        ('.jpg', '.jpeg', '.png')
    ):

        try:

            student_name = extract_name(file)

            path = os.path.join(
                DATASET_PATH,
                file
            )

            embedding = get_embedding(path)

            # Create student record if not exists
            if student_name not in database:

                database[student_name] = {

                    "Enrollment_No": "N/A",

                    "Branch": "N/A",

                    "Semester": "N/A",

                    "Section": "N/A",

                    "Profile_Image": file,

                    "Embeddings": []

                }

            database[
                student_name
            ]["Embeddings"].append(
                embedding
            )

            count += 1

            print("Added:", student_name)

        except Exception as e:

            print("\nFAILED:", file)

            print("ERROR:", str(e))

print("\nStudents Added:", count)

1/1 [==============================] - 6s 6s/step
Added: PRIYAL MARU
1/1 [==============================] - 0s 258ms/step
Added: NANDINI SINGH
1/1 [==============================] - 0s 197ms/step
Added: AADARSH SAHU
1/1 [==============================] - 0s 379ms/step
Added: ANSHITA SONI
1/1 [==============================] - 0s 329ms/step
Added: LABHANSHU SAHU
1/1 [==============================] - 0s 180ms/step
Added: ALAQMAR KANCHWALA
1/1 [==============================] - 0s 163ms/step
Added: SALONI DUBEY
1/1 [==============================] - 0s 210ms/step
Added: PRIYANSHI SEN
1/1 [==============================] - 0s 100ms/step
Added: MAYANK SAHU
1/1 [==============================] - 0s 164ms/step
Added: ANSHITA KUMBHARE
1/1 [==============================] - 0s 169ms/step
Added: SHIVANGI SHARMA
1/1 [==============================] - 0s 186ms/step
Added: NAMAN GAUR
1/1 [==============================] - 0s 91ms/step
Added: HEMANT SHARMA
1/1 [==============================] - 0s 

In [ ]:
#Block 8: Save Embeddings
import pickle

with open(
    "student_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(database, f)

print("Database Saved")

Database Saved


In [ ]:

# Block 9: Load Embeddings

import os
import pickle

EMBEDDING_FILE = "student_embeddings.pkl"

if os.path.exists(EMBEDDING_FILE):

    with open(EMBEDDING_FILE, "rb") as f:
        database = pickle.load(f)

    print("Database Loaded Successfully")
    print("Total Students:", len(database))

else:

    print("Embedding file not found!")
    print("Run Block 7 and Block 8 first.")

Database Loaded Successfully
Total Students: 175


In [ ]:
# ==================================
# Webcam Function For New Student//fix crash webcam
# ==================================

from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def take_student_photo(
    filename='new_student.jpg',
    quality=0.9
):

    js = Javascript('''
    async function takePhoto(quality) {

      const div = document.createElement('div');
      const button = document.createElement('button');

      button.textContent = 'Capture Student';

      div.appendChild(button);
      document.body.appendChild(div);

      const video = document.createElement('video');

      const stream =
      await navigator.mediaDevices.getUserMedia({
        video:true
      });

      document.body.appendChild(video);

      video.srcObject = stream;

      await video.play();

      await new Promise(
        resolve => button.onclick = resolve
      );

      const canvas =
      document.createElement('canvas');

      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      canvas.getContext('2d')
            .drawImage(video,0,0);

      stream.getTracks()
            .forEach(track => track.stop());

      video.remove();
      button.remove();

      return canvas.toDataURL(
        'image/jpeg',
        quality
      );
    }
    ''')

    display(js)

    data = eval_js(
        f'takePhoto({quality})'
    )

    binary = b64decode(
        data.split(',')[1]
    )

    with open(filename,'wb') as f:
        f.write(binary)

    return filename

In [ ]:
# ==================================
# Block 9A : Add New Student
# ==================================

print("\n1. Add New Student")
print("2. Skip / Continue")

main_choice = input(
    "\nEnter Choice (1/2): "
)

if main_choice == "2":

    print(
        "\nSkipping student enrollment..."
    )

else:

    print("\n1. Upload Student Image")
    print("2. Use Webcam")

    choice = input(
        "\nEnter Choice (1/2): "
    )

    if choice == "1":

        uploaded = files.upload()

        file_name = list(
            uploaded.keys()
        )[0]

        image_path = (
            "/content/" +
            file_name
        )

    elif choice == "2":

        image_path = take_student_photo()

    else:

        raise ValueError(
            "Invalid Choice"
        )

    student_name = input(
        "\nEnter Student Name : "
    ).strip().upper()

    import shutil
    import glob
    import os

    extension = os.path.splitext(
        image_path
    )[1]

    existing_images = glob.glob(
        os.path.join(
            DATASET_PATH,
            f"{student_name}_*"
        )
    )

    image_count = len(
        existing_images
    ) + 1

    new_filename = (
        f"{student_name}_{image_count}"
        f"{extension}"
    )

    save_path = os.path.join(
        DATASET_PATH,
        new_filename
    )

    shutil.copy(
        image_path,
        save_path
    )

    print(
        "\nImage Saved:"
    )

    print(
        save_path
    )

    enrollment_no = input(
        "Enrollment No (Enter for N/A): "
    ).strip()

    branch = input(
        "Branch (Enter for N/A): "
    ).strip()

    semester = input(
        "Semester (Enter for N/A): "
    ).strip()

    section = input(
        "Section (Enter for N/A): "
    ).strip()

    if enrollment_no == "":
        enrollment_no = "N/A"

    if branch == "":
        branch = "N/A"

    if semester == "":
        semester = "N/A"

    if section == "":
        section = "N/A"

    embedding = get_embedding(
        save_path
    )

    if student_name not in database:

        database[
            student_name
        ] = {

            "Enrollment_No":
            enrollment_no,

            "Branch":
            branch,

            "Semester":
            semester,

            "Section":
            section,

            "Profile_Image":
            new_filename,

            "Embeddings": []

        }

    database[
        student_name
    ]["Embeddings"].append(
        embedding
    )

    with open(
        "student_embeddings.pkl",
        "wb"
    ) as f:

        pickle.dump(
            database,
            f
        )

    print(
        "\nStudent Added Successfully"
    )

    print(
        "Name :",
        student_name
    )

    print(
        "Total Embeddings :",
        len(
            database[
                student_name
            ]["Embeddings"]
        )
    )

    print(
        "Total Students :",
        len(database)
    )


1. Add New Student
2. Skip / Continue


In [ ]:
#block  10 , 11 deleted ---

class group photo detection.

In [ ]:
# ==================================
# Block 12 : Detect All Faces
# ==================================

def detect_all_faces(img_path):

    img = cv2.imread(img_path)

    if img is None:
        raise ValueError("Cannot read image")

    rgb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    faces = detector.detect_faces(rgb)

    return rgb, faces

In [ ]:
# ==================================
# Block 13 : Recognize Face Crop
# ==================================

from sklearn.metrics.pairwise import cosine_similarity

def recognize_face_crop(face_crop):

    face_crop = cv2.resize(
        face_crop,
        (160,160)
    )

    embedding = embedder.embeddings(

        np.expand_dims(
            face_crop,
            axis=0
        )

    )[0]

    best_match = "Unknown"
    best_score = -1

    for student, info in database.items():

        if "Embeddings" not in info:
            continue

        for stored_embedding in info["Embeddings"]:

            score = cosine_similarity(

                [embedding],
                [stored_embedding]

            )[0][0]

            if score > best_score:

                best_score = score
                best_match = student

    return best_match, float(best_score)

print("Block 13 Loaded Successfully")

In [ ]:
# ==================================
# Unknown Face Manager
# ==================================

import os
import pickle
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

UNKNOWN_FOLDER = "/content/drive/MyDrive/Attendance/unknown_faces"

os.makedirs(
    UNKNOWN_FOLDER,
    exist_ok=True
)

UNKNOWN_EMBEDDINGS = os.path.join(
    UNKNOWN_FOLDER,
    "unknown_embeddings.pkl"
)

UNKNOWN_LOG = os.path.join(
    UNKNOWN_FOLDER,
    "unknown_log.csv"
)


# Load old unknown embeddings

if os.path.exists(UNKNOWN_EMBEDDINGS):

    with open(
        UNKNOWN_EMBEDDINGS,
        "rb"
    ) as f:

        unknown_db = pickle.load(f)

else:

    unknown_db = {}


def save_unknown_face(face_crop):

    global unknown_db

    face_resized = cv2.resize(
        face_crop,
        (160,160)
    )

    emb = embedder.embeddings(
        np.expand_dims(
            face_resized,
            axis=0
        )
    )[0]

    # Duplicate Unknown Check

    for file_name, old_emb in unknown_db.items():

        score = cosine_similarity(
            [emb],
            [old_emb]
        )[0][0]

        if score >= 0.85:

            print(
                "Duplicate Unknown Ignored"
            )

            return None

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    filename = (
        f"unknown_{timestamp}.jpg"
    )

    save_path = os.path.join(
        UNKNOWN_FOLDER,
        filename
    )

    cv2.imwrite(
        save_path,
        cv2.cvtColor(
            face_crop,
            cv2.COLOR_RGB2BGR
        )
    )

    unknown_db[
        filename
    ] = emb

    with open(
        UNKNOWN_EMBEDDINGS,
        "wb"
    ) as f:

        pickle.dump(
            unknown_db,
            f
        )

    log_row = pd.DataFrame([{
        "Date":
        datetime.now().strftime(
            "%Y-%m-%d"
        ),

        "Time":
        datetime.now().strftime(
            "%H:%M:%S"
        ),

        "Filename":
        filename
    }])

    if os.path.exists(
        UNKNOWN_LOG
    ):

        log_row.to_csv(
            UNKNOWN_LOG,
            mode="a",
            header=False,
            index=False
        )

    else:

        log_row.to_csv(
            UNKNOWN_LOG,
            index=False
        )

    print(
        "Unknown Saved:",
        filename
    )

    return filename

In [ ]:
# ==================================
# Block 14 : Group Photo Attendance
# Upload OR Webcam
# ==================================

from base64 import b64decode
import cv2
from google.colab import files
from google.colab.output import eval_js
from IPython.display import Javascript, display


def take_group_photo(filename="group_photo.jpg", quality=0.9):

    js = Javascript("""
    async function takePhoto(quality) {

      const div = document.createElement('div');
      const button = document.createElement('button');

      button.textContent = 'Capture Group Photo';

      div.appendChild(button);
      document.body.appendChild(div);

      const video = document.createElement('video');

      const stream =
      await navigator.mediaDevices.getUserMedia({
        video:true
      });

      document.body.appendChild(video);

      video.srcObject = stream;

      await video.play();

      await new Promise(
        resolve => button.onclick = resolve
      );

      const canvas =
      document.createElement('canvas');

      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      canvas
        .getContext('2d')
        .drawImage(
          video,
          0,
          0
        );

      stream.getTracks()
            .forEach(
              track => track.stop()
            );

      video.remove();
      button.remove();

      return canvas.toDataURL(
        'image/jpeg',
        quality
      );
    }
    """)

    display(js)

    data = eval_js(f"takePhoto({quality})")

    binary = b64decode(
        data.split(",")[1]
    )

    with open(filename, "wb") as f:
        f.write(binary)

    return filename


print("\n1. Upload Group Photo")
print("2. Capture From Webcam")

choice = input(
    "\nEnter Choice (1/2): "
)

if choice == "1":

    uploaded = files.upload()

    group_file = list(
        uploaded.keys()
    )[0]

    group_path = (
        "/content/" +
        group_file
    )

elif choice == "2":

    group_path = take_group_photo()

else:

    raise ValueError(
        "Invalid Choice"
    )


image, faces = detect_all_faces(
    group_path
)

present_students = []
attendance_records = []
possible_matches = []
unknown_count = 0

print(
    f"\nTotal Faces Detected: {len(faces)}"
)

for face_data in faces:

    x, y, w, h = face_data["box"]

    x = max(0, x)
    y = max(0, y)

    if w <= 0 or h <= 0:
        continue

    face_crop = image[
        y:y+h,
        x:x+w
    ]

    if face_crop.size == 0:
        continue

    try:

        name, score = recognize_face_crop(
            face_crop
        )

        print(
            "Prediction:",
            name,
            "Score:",
            round(score, 4)
        )

        # ==========================
        # HIGH CONFIDENCE
        # ==========================
        if score >= 0.80:

            present_students.append(
                name
            )

            student_info = database[
                name
            ]

            attendance_records.append(

                {

                    "Enrollment_No":
                    student_info[
                        "Enrollment_No"
                    ],

                    "Student_Name":
                    name,

                    "Branch":
                    student_info[
                        "Branch"
                    ],

                    "Semester":
                    student_info[
                        "Semester"
                    ],

                    "Section":
                    student_info[
                        "Section"
                    ],

                    "Profile_Image":
                    student_info[
                        "Profile_Image"
                    ],

                    "Confidence":
                    round(
                        float(score),
                        4
                    ),

                    "Status":
                    "Present",

                    "Attendance_Mode":
                    "Group Photo"

                }

            )

            cv2.rectangle(

                image,

                (x, y),

                (x + w, y + h),

                (0, 255, 0),

                3

            )

            cv2.putText(

                image,

                f"{name} ({score:.2f})",

                (x, y - 10),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.7,

                (0, 255, 0),

                2

            )

        # ==========================
        # POSSIBLE MATCH
        # ==========================
        elif score >= 0.65:

            possible_matches.append(
                (name, score)
            )

            cv2.rectangle(

                image,

                (x, y),

                (x + w, y + h),

                (0, 255, 255),

                3

            )

            cv2.putText(

                image,

                f"Possible: {name}",

                (x, y - 10),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.6,

                (0, 255, 255),

                2

            )

        # ==========================
        # UNKNOWN
        # ==========================
        else:

            save_unknown_face(
                face_crop
            )

            unknown_count += 1

            cv2.rectangle(

                image,

                (x, y),

                (x + w, y + h),

                (255, 0, 0),

                3

            )

            cv2.putText(

                image,

                "Unknown",

                (x, y - 10),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.7,

                (255, 0, 0),

                2

            )

    except Exception as e:

        print(
            "Face Error:",
            str(e)
        )

        continue


# Remove duplicate attendance
present_students = sorted(
    set(present_students)
)

print("\n========== SUMMARY ==========")

print(
    "Total Faces Detected :",
    len(faces)
)

print(
    "Recognized Students :",
    len(present_students)
)

print(
    "Possible Matches :",
    len(possible_matches)
)

print(
    "Unknown Faces :",
    unknown_count
)

In [ ]:
import matplotlib.pyplot as plt

# ==================================
# Block 15 : Display Attendance
# ==================================

plt.figure(figsize=(12,12))

plt.imshow(image)

plt.axis("off")

plt.title(
    f"Detected Students : {len(present_students)}"
)

plt.show()

print("\nPresent Students:\n")

for student in sorted(
    set(present_students)
):
    print(student)

In [ ]:
# ==================================
# Block 16 : Attendance Manager
# Save Attendance + Master Record
# + Attendance Report
# ==================================

import os
import pandas as pd
from datetime import datetime

ATTENDANCE_FOLDER = (
    "/content/drive/MyDrive/Attendance"
)

os.makedirs(
    ATTENDANCE_FOLDER,
    exist_ok=True
)

# ==========================
# Add Date & Time
# ==========================

current_date = datetime.now().strftime(
    "%Y-%m-%d"
)

current_time = datetime.now().strftime(
    "%H:%M:%S"
)

for record in attendance_records:

    record["Date"] = current_date
    record["Time"] = current_time

# ==========================
# Session Attendance
# ==========================

columns = [

    "Enrollment_No",

    "Student_Name",

    "Branch",

    "Semester",

    "Section",

    "Profile_Image",

    "Confidence",

    "Status",

    "Attendance_Mode",

    "Date",

    "Time"

]

attendance_df = pd.DataFrame(

    attendance_records,

    columns=columns

)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

session_file = os.path.join(

    ATTENDANCE_FOLDER,

    f"attendance_{timestamp}.csv"

)

attendance_df.to_csv(

    session_file,

    index=False

)

# ==========================
# Master Attendance
# ==========================

MASTER_FILE = os.path.join(

    ATTENDANCE_FOLDER,

    "master_attendance.csv"

)

if os.path.exists(
    MASTER_FILE
):

    master_df = pd.read_csv(

        MASTER_FILE,

        keep_default_na=False

    )

else:

    master_df = pd.DataFrame(
        columns=columns
    )

attendance_df = attendance_df.reindex(
    columns=columns
)

master_df = master_df.reindex(
    columns=columns
)

master_df = pd.concat(

    [

        master_df,

        attendance_df

    ],

    ignore_index=True

)

master_df.to_csv(

    MASTER_FILE,

    index=False

)

# ==========================
# Attendance Report
# ==========================

report = []

total_classes = master_df[
    "Date"
].nunique()

for student in master_df[
    "Student_Name"
].unique():

    student_df = master_df[

        master_df[
            "Student_Name"
        ] == student

    ]

    info = student_df.iloc[0]

    present_count = len(
        student_df
    )

    if total_classes > 0:

        attendance_percent = round(

            (
                present_count
                /
                total_classes
            ) * 100,

            2

        )

    else:

        attendance_percent = 0.0

    report.append(

        {

            "Enrollment_No":
            info[
                "Enrollment_No"
            ],

            "Student_Name":
            student,

            "Branch":
            info[
                "Branch"
            ],

            "Semester":
            info[
                "Semester"
            ],

            "Section":
            info[
                "Section"
            ],

            "Profile_Image":
            info[
                "Profile_Image"
            ],

            "Classes_Attended":
            present_count,

            "Total_Classes":
            total_classes,



        }

    )

report_df = pd.DataFrame(
    report
)



# ==========================
# Save Attendance Report
# ==========================

REPORT_FILE = os.path.join(

    ATTENDANCE_FOLDER,

    "attendance_report.csv"

)

report_df.to_csv(

    REPORT_FILE,

    index=False

)

# ==========================
# Summary
# ==========================

print(
    "\n========== ATTENDANCE SUMMARY =========="
)

print(
    "Recognized Students :",
    len(attendance_df)
)

print(
    "Session CSV Saved :"
)

print(
    session_file
)

print(
    "\nMaster Attendance Updated"
)

print(
    "Total Records :",
    len(master_df)
)

print(
    "\nAttendance Report Updated"
)

print(
    REPORT_FILE
)

print(
    "\nTop Attendance Students:"
)

display(
    report_df.head()
)